###Build Constructors Dimension
1. Read silver constructors table
2. Read silver ref_nationality_region table
3. Join the data from constructors with ref_nationality_region using nationality
4. Select the required columns
5. Write the transformed data to gold dim_constructors table

In [0]:
%run ../00-Common/01.environment-config

In [0]:
from pyspark.sql import functions as f

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_constructors"

In [0]:
constructors_df  = spark.table(f"{catalog_name}.{silver_schema}.constructors")
ref_nationality_region_df = spark.table(f"{catalog_name}.{gold_schema}.ref_nationality_region")

In [0]:
dim_constructors_df = (
    constructors_df
        .join(
            ref_nationality_region_df,
            constructors_df.nationality == ref_nationality_region_df.nationality,
            "left"
            )
        .select(
            constructors_df.constructor_id,
            constructors_df.name,
            constructors_df.nationality,
            ref_nationality_region_df.region.alias("nationality_region")
        )
)

In [0]:
display(dim_constructors_df)

In [0]:
(
    dim_constructors_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
)

In [0]:
display(spark.table(target_table))